[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_14/3_stats.ipynb)

# Day 14: Statistics, A/B testing and product (mock exam)

**Mock exam rules.** One timer for the whole notebook: **30 minutes**. Run the setup cell, then start the timer. Work through Q1 to Q5 in order; skip and come back if stuck. Open hints and solutions **only after time is up**. For each question give the number *and* the sentence for the PM; in a real exam the sentence is half of the grade. Afterwards, score yourself: 2 points per question (1 for the number or the method, 1 for the interpretation).

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Do smokers tip differently? | medium | 5 |
| Q2 | Contract type and churn | medium | 5 |
| Q3 | Size a test on rounds played | medium | 6 |
| Q4 | Would daily checking have fooled us? | medium | 6 |
| Q5 | Case: should Cookie Cats move the gate? | medium | 8 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `tips` (244 restaurant bills, real): `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`.

- `telco` (IBM Telco churn, real, 7,043 customers): `tenure` (months), `Contract`, `MonthlyCharges`, `TotalCharges` (text with blanks), service columns such as `InternetService`, `TechSupport`, and `Churn` (Yes/No).

- `cats` (Cookie Cats, a real mobile-game A/B test, 90,189 players): `userid`, `version` (gate_30 = control, gate_40 = treatment: the first gate moved from level 30 to level 40), `sum_gamerounds` (rounds played in the first 14 days after install), `retention_1`, `retention_7` (True if the player came back 1 / 7 days after install).

- `ab` (Udacity e-commerce landing page test, real, 294,478 rows): `user_id`, `timestamp`, `group` (control / treatment), `landing_page` (old_page / new_page), `converted` (0/1). It has some rows where group and page do not match and some repeated users.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
tips = pd.read_csv(data_path("tips.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"))
telco = pd.read_csv(data_path("telco_churn.csv", "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"))
cats = pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv"))
ab = pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv"))
print("loaded:", "tips", tips.shape, "telco", telco.shape, "cats", cats.shape, "ab", ab.shape)

---
### Q1. Do smokers tip differently?

*medium, about 5 min*

Using `tips`, define `pct = tip / total_bill`. Do smokers and non-smokers tip a different percentage?

(a) Mean and standard deviation of `pct` per group, a Welch t-test p-value, and a 95% CI for the difference in means (smokers minus non-smokers; use the Welch-Satterthwaite df or 1.96, say which).

(b) Say it to a PM.

Follow-up: smokers' standard deviation is twice as large. Why Welch and not Student's t-test? What does a single 71% tip do to the result?

In [ ]:
tips["pct"] = tips.tip / tips.total_bill
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a numeric outcome, two independent groups, unequal spread. Method: Welch two-sample t-test.

</details>

<details><summary><b>Hint 2</b></summary>

1. `a, b = pct[smoker=='Yes'], pct[smoker=='No']`. 2. `stats.ttest_ind(a, b, equal_var=False)`. 3. `se = sqrt(var_a/n_a + var_b/n_b)`, CI = diff plus or minus 1.96 * se.

</details>

<details><summary><b>Solution</b></summary>

```python
tips["pct"] = tips.tip / tips.total_bill
a = tips.pct[tips.smoker == "Yes"]
b = tips.pct[tips.smoker == "No"]
print(tips.groupby("smoker").pct.agg(["mean", "std", "count"]).round(4))
#           mean     std  count
# No      0.1593  0.0399    151
# Yes     0.1632  0.0851     93
res = stats.ttest_ind(a, b, equal_var=False)
diff = a.mean() - b.mean()
se = np.sqrt(a.var() / len(a) + b.var() / len(b))
print(f"diff {diff * 100:.2f} pp, p {res.pvalue:.2f}, CI [{(diff - 1.96 * se) * 100:.2f}, {(diff + 1.96 * se) * 100:.2f}] pp")
# diff 0.39 pp, p 0.68, CI [-1.46, 2.23] pp
```

**Why:** Both groups tip about 16% of the bill; the 0.39 pp gap is far inside the noise (p = 0.68) and the CI runs from -1.46 to +2.23 pp (1.96 used; the Welch df is about 117, which would widen it very slightly).

**Say it to a PM:** Smokers and non-smokers tip about the same share of the bill (around 16%); with 244 bills we cannot rule out a gap of up to about 2 points either way.

**Follow-ups:**

- *Why Welch?* Student's test pools the variances, which is wrong when they differ (0.085 vs 0.040) and the group sizes differ; Welch does not assume equal variances and costs almost nothing when they are equal, so use it by default.
- *The 71% tip?* It inflates the smokers' mean and standard deviation. Check robustness with a Mann-Whitney test (p = 0.56 here) or by excluding it with a stated rule; the conclusion does not change.

**Complexity:** O(n).

**Common mistakes:** Testing `tip` instead of tip percentage (bigger bills get bigger tips). Using a paired test (the groups are different bills).

**Learn more:** [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests), [stats-choosing-a-test](https://mahsa-agz.github.io/ds-handbook/#stats-choosing-a-test)

</details>

---
### Q2. Contract type and churn

*medium, about 5 min*

Using `telco`:

(a) Churn rate per `Contract` type. Test whether churn and contract type are related, and give an effect size (Cramer's V = `sqrt(chi2 / (n * (min(rows, cols) - 1)))`).

(b) Say it to a PM, careful with the wording.

(c) Rapid fire, name the test: (1) day-7 retention, treatment vs control; (2) revenue per user, very skewed, 2 arms; (3) the same users rate the old and the new design; (4) clicks / views per user, randomised by user; (5) conversion across 4 arms.

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: two categorical variables. Method: chi-square test of independence on the contingency table; Cramer's V for the size.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tab = pd.crosstab(telco.Contract, telco.Churn)`. 2. `stats.chi2_contingency(tab)` returns chi2, p, dof, expected. 3. V with min(3, 2) - 1 = 1.

</details>

<details><summary><b>Solution</b></summary>

```python
tab = pd.crosstab(telco.Contract, telco.Churn)
print((tab["Yes"] / tab.sum(axis=1)).round(3).to_dict())
# {'Month-to-month': 0.427, 'One year': 0.113, 'Two year': 0.028}
chi2, p, dof, _ = stats.chi2_contingency(tab)
v = np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))
print(f"chi2 {chi2:.0f}, dof {dof}, p {p:.1e}, Cramer's V {v:.2f}")
# chi2 1185, dof 2, p 5.9e-258, Cramer's V 0.41
```

**Why:** Month-to-month customers churn at 42.7%, fifteen times the 2.8% of two-year customers. The chi-square test says the link is not chance, and V = 0.41 says it is strong (above 0.3 is usually called large for a 2-column table). The data is observational, so this is association, not proof that contracts cause loyalty: committed customers choose long contracts.

**Say it to a PM:** Churn is much higher on month-to-month plans (43% vs 3% on two-year plans); that tells us where to focus, but to know whether pushing people to longer contracts *reduces* churn we would need an experiment, for example a discount offer test.

**Follow-ups:**

- *Rapid fire answers* (1) two-proportion z-test (or chi-square 2x2); (2) Welch t-test on capped values, bootstrap, or Mann-Whitney if a distribution shift is the question; (3) paired t-test or Wilcoxon signed-rank; (4) delta method (or bootstrap by user) for the ratio, because views within a user are correlated; (5) chi-square on the 4x2 table, then pairwise tests with a multiple-testing correction.

**Complexity:** O(n).

**Common mistakes:** Reporting only the p-value (with 7,043 rows almost anything is significant; give rates and V). Saying 'contracts reduce churn'.

**Learn more:** [stats-choosing-a-test](https://mahsa-agz.github.io/ds-handbook/#stats-choosing-a-test), [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests)

</details>

---
### Q3. Size a test on rounds played

*medium, about 6 min*

The game team wants a new test with primary metric `sum_gamerounds` (use Cookie Cats gate_30 as the baseline). They want to detect a change of 2 rounds per player.

(a) Players per group for alpha 0.05 (two-sided), power 0.8: `n = 2 * (z_a + z_b)^2 * sd^2 / delta^2`. Use the raw standard deviation.

(b) Repeat after capping the metric at its 99th percentile (winsorizing). What does capping change about what you measure?

(c) Say it to a PM.

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: sample size for a mean difference, and a heavy tail. Method: power formula for two means; variance reduction by capping.

</details>

<details><summary><b>Hint 2</b></summary>

1. `x = cats.sum_gamerounds[cats.version=='gate_30']`. 2. `sd = x.std()`. 3. `cap = x.quantile(0.99)`, `x.clip(upper=cap).std()`. 4. Plug both into the formula.

</details>

<details><summary><b>Solution</b></summary>

```python
x = cats.loc[cats.version == "gate_30", "sum_gamerounds"]
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
def n_means(sd, delta=2):
    return int(np.ceil(2 * z ** 2 * sd ** 2 / delta ** 2))
cap = x.quantile(0.99)
sd_raw, sd_cap = x.std(), x.clip(upper=cap).std()
print(f"sd raw {sd_raw:.1f}, cap at {cap:.0f} rounds, sd capped {sd_cap:.1f}")
print("n per group raw:", n_means(sd_raw), " capped:", n_means(sd_cap))
# sd raw 256.7, cap at 493 rounds, sd capped 84.5
# n per group raw: 258634  capped: 28002
```

**Why:** One player with 49,854 rounds makes the raw standard deviation 256.7; capping at the 99th percentile (493 rounds) brings it to 84.5, and since n grows with sd squared, the needed sample falls more than 9 times (258,634 to 28,002 per group). The capped metric is a slightly different question: it ignores how far the top 1% go beyond 493 rounds. Set the cap from pre-test data and write it in the plan.

**Say it to a PM:** If we cap extreme players at about 500 rounds, a 2 round change becomes detectable with about 28,000 players per group instead of about 260,000, at the cost of not measuring changes among the top 1% of players.

**Follow-ups:**

- *Other ways to cut variance?* CUPED with pre-period rounds (day 16), a log transform (changes the estimand to a ratio), or a triggered analysis.

**Complexity:** O(n).

**Common mistakes:** Using the per-group formula without the factor 2. Choosing the cap after seeing test results. Capping at a different value in each arm.

**Learn more:** [stats-power-mde](https://mahsa-agz.github.io/ds-handbook/#stats-power-mde), [stats-cuped](https://mahsa-agz.github.io/ds-handbook/#stats-cuped)

</details>

---
### Q4. Would daily checking have fooled us?

*medium, about 6 min*

Use the cleaned `ab` table (drop group/page mismatches, keep the first row per user).

(a) For each calendar day, compute the two-proportion z-test p-value on all data **up to and including** that day. On how many days is p < 0.05? What is the smallest p-value and on which day?

(b) What is the final p-value?

(c) Say it to a PM.

In [ ]:
ok = (ab.group == "treatment") == (ab.landing_page == "new_page")
clean = ab[ok].sort_values("timestamp").drop_duplicates("user_id")
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a p-value tracked over time. Pitfall: peeking. Compute the cumulative test day by day.

</details>

<details><summary><b>Hint 2</b></summary>

1. `day = pd.to_datetime(clean.timestamp).dt.date`. 2. Daily sums and counts per group with `groupby([day, 'group'])`, then `cumsum()`. 3. z-test per row.

</details>

<details><summary><b>Solution</b></summary>

```python
ok = (ab.group == "treatment") == (ab.landing_page == "new_page")
clean = ab[ok].sort_values("timestamp").drop_duplicates("user_id")
clean["day"] = pd.to_datetime(clean.timestamp).dt.date
d = clean.groupby(["day", "group"]).converted.agg(["sum", "count"]).unstack().cumsum()
x_c, x_t = d["sum"]["control"], d["sum"]["treatment"]
n_c, n_t = d["count"]["control"], d["count"]["treatment"]
pool = (x_c + x_t) / (n_c + n_t)
z = (x_t / n_t - x_c / n_c) / np.sqrt(pool * (1 - pool) * (1 / n_t + 1 / n_c))
p = pd.Series(2 * stats.norm.sf(abs(z)), index=z.index)
print("days:", len(p), " days with p < 0.05:", (p < 0.05).sum())
print("smallest p:", round(p.min(), 3), "on", p.idxmin(), " final p:", round(p.iloc[-1], 3))
# days: 23  days with p < 0.05: 0
# smallest p: 0.127 on 2017-01-05  final p: 0.19
```

**Why:** Over 23 calendar days the cumulative p-value never went below 0.05; its lowest point was 0.127 on 2017-01-05 (when little data had arrived) and it ended at 0.19. Here peeking would not have produced a false win, but that is luck, not a property of the method: a p-value path that wanders up and down is normal under the null, which is why one pre-registered look (or a sequential method) is needed.

**Say it to a PM:** Checking every day, we would have seen the p-value move around, but it never gave a reliable signal; the final read after the planned 3 weeks says there is no meaningful difference.

**Follow-ups:**

- *If it had dipped below 0.05 on one day?* Under daily peeking the false positive rate is not 5% but about 20 to 25% for 20 looks (day 13), so a single dip is not evidence. Use the planned end date or an alpha spending boundary.

**Complexity:** O(n log n) for the sort, then O(days).

**Common mistakes:** Using daily (not cumulative) data and calling it peeking. Forgetting the first and last days are partial days.

**Learn more:** [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls), [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis)

</details>

---
### Q5. Case: should Cookie Cats move the gate?

*medium, about 8 min*

You have this readout from the Cookie Cats test (gate_30 = control, gate_40 = treatment):

| metric | gate_30 | gate_40 | diff | p |
|---|---|---|---|---|
| players | 44,700 | 45,489 | | SRM p = 0.0086 |
| day-1 retention | 44.82% | 44.23% | -0.59 pp | 0.074 |
| day-7 retention | 19.02% | 18.20% | -0.82 pp | 0.0016 |
| mean rounds | 52.5 | 51.3 | -1.2 | CI includes 0 |

The PM says: 'Day-1 is flat and rounds are flat, the later gate gives players more free play, let's ship gate_40.' Give your recommendation in 5 to 7 bullets: what you trust, what you check first, the decision, and what you would do next. Then say it to the PM in two sentences.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Framework: data quality first (SRM), then the primary metric, then guardrails and mechanism, then the decision with its risk, then next steps.

</details>

<details><summary><b>Hint 2</b></summary>

1. SRM p = 0.0086: investigate before trusting. 2. Day-7 retention is the long-term metric and is clearly down. 3. 'Flat' day-1 is not zero (CI to -1.24 pp). 4. Mechanism: the gate forces a break that may build anticipation. 5. Decide, and say what would change your mind.

</details>

<details><summary><b>Solution</b></summary>

- **Check data quality first.** The split is 50.44 / 49.56 with SRM p = 0.0086. Before any decision, check
  assignment logs, platform/version mix, and whether one arm lost users through logging. If a cause is found and it
  is unrelated to engagement, the metrics can still be used; if not, re-run or treat the result with caution.
- **Primary metric:** day-7 retention is closest to long-term value (revenue, lifetime). It dropped 0.82 pp (4.3%
  relative), p = 0.0016, CI about -1.33 to -0.31 pp. That is a clear harm, not noise.
- **Day-1 is not "flat":** -0.59 pp with a CI from -1.24 to +0.06 pp. Not significant, but it points the same way.
- **Rounds:** the mean is dominated by one player with 49,854 rounds; without that player the difference is about
  0. So more free play did not translate into more play.
- **Mechanism (plausible story):** the gate at level 30 forces a pause that keeps the game fresh (hedonic
  adaptation); moving it to 40 lets players burn out faster. This is a hypothesis, not proven by this test.
- **Decision:** keep gate_30. Shipping gate_40 risks about 1 in 23 day-7 retained players (0.82 / 19.02).
- **Next:** fix or explain the SRM; if the team still wants a later gate, test intermediate options (gate at 35) and
  track day-14 and day-30 retention and revenue.

**Say it to a PM:** "The later gate loses about 4% of the players who would still be playing after a week, and that
drop is very unlikely to be chance, so I recommend we keep the gate at level 30. Before we close this, I want to check
why the two groups are slightly unequal in size, but that is unlikely to reverse the result."

**Why:** A good case answer is ordered (trust, then metrics, then decision), uses the primary metric agreed before the test instead of the most convenient one, and corrects 'flat' into 'not significant, here is the CI'.

**Common mistakes:** Agreeing with the PM because two of three metrics are 'flat'. Ignoring the SRM flag. Calling the rounds difference significant because of the outlier.

**Learn more:** [stats-launch-decisions](https://mahsa-agz.github.io/ds-handbook/#stats-launch-decisions), [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_14/4_ai.ipynb)